# Demo-1 Task-2: Automobile Price Prediction (DNN + Linear Regression)

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

auto = pd.read_csv("Automobile_price_data_Raw_set.csv")

# Features and target exactly as task says
feature_cols = ["make", "body-style", "wheel-base", "engine-size", "horsepower", "peak-rpm", "highway-mpg"]
target_col = "price"

auto = auto[feature_cols + [target_col]].dropna()

auto.head()

,make,body-style,wheel-base,engine-size,horsepower,peak-rpm,highway-mpg,price
0,alfa-romero,convertible,88.6,130,111.0,5000.0,27,13495.0
1,alfa-romero,convertible,88.6,130,111.0,5000.0,27,16500.0
2,alfa-romero,hatchback,94.5,152,154.0,5000.0,26,16500.0
3,audi,sedan,99.8,109,102.0,5500.0,30,13950.0
4,audi,sedan,99.4,136,115.0,5500.0,22,17450.0


In [2]:
auto.shape

(194, 8)

In [3]:
# Split features/target
X = auto[feature_cols]
y = auto[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Categorical and numeric columns
cat_cols = ["make", "body-style"]
num_cols = ["wheel-base", "engine-size", "horsepower", "peak-rpm", "highway-mpg"]

# Preprocessing
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols)
    ]
)

X_train.shape, X_test.shape

((155, 7), (39, 7))

In [7]:
from sklearn.linear_model import LinearRegression

# Build pipeline: preprocessing + linear regression
linreg_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LinearRegression())
])

# Train
linreg_model.fit(X_train, y_train)

# Evaluate
y_pred_lr = linreg_model.predict(X_test)
mse_lr = mean_squared_error(y_test, y_pred_lr)

mse_lr

print("Linear Regression Test MSE:", mse_lr)
print("Linear Regression R^2:", linreg_model.score(X_test, y_test))


Linear Regression Test MSE: 7781562.306999621
Linear Regression R^2: 0.9324245728334127


In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Build DNN model
dnn_model = Sequential([
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1)
])

dnn_model.compile(
    optimizer="adam",
    loss="mse"
)

# Preprocess data first
X_train_prep = preprocess.fit_transform(X_train)
X_test_prep = preprocess.transform(X_test)

# Train
dnn_model.fit(
    X_train_prep, y_train,
    epochs=50,
    batch_size=16,
    verbose=0
)

# Evaluate
y_pred_dnn = dnn_model.predict(X_test_prep, verbose=0)
mse_dnn = mean_squared_error(y_test, y_pred_dnn)

mse_dnn

199074155.21261197

In [6]:
# Required input sample
sample = pd.DataFrame([{
    "make": "audi",
    "body-style": "hatchback",
    "wheel-base": 99.5,
    "engine-size": 131,
    "horsepower": 160,
    "peak-rpm": 5500,
    "highway-mpg": 22
}])

# Linear Regression prediction
pred_lr = linreg_model.predict(sample)[0]

# DNN prediction
sample_prep = preprocess.transform(sample)
pred_dnn = dnn_model.predict(sample_prep, verbose=0)[0][0]

pred_lr, pred_dnn

(np.float64(20348.837855373753), np.float32(4468.2344))

### Demo-1 Task-2 Results (Summary)


- Linear Regression Test MSE: 7,781,562.31
- Linear Regression R² Score: 0.93
- DNN Test MSE: 199,074,155.21
Prediction for [audi, hatchback, 99.5, 131, 160, 5500, 22]:
  - Linear Regression: 20348.84
  - DNN: 4468.23